# 🌿 AgroAI - PlantVillage Full Dataset Training Pipeline (GPU Accelerated)

This notebook downloads the complete **PlantVillage Dataset (~54,303 images across 38+ crop classes)** and trains a **MobileNetV3** vision backbone using **PyTorch with GPU Acceleration**.

### Recommended Runtime:
Go to **Runtime > Change runtime type > T4 GPU** (Free on Google Colab).

In [ ]:
# 1. Verify GPU acceleration
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (switch to T4 GPU in Runtime settings for 10x faster training).")

In [ ]:
# 2. Download PlantVillage Dataset (Direct Archive Mirror)
!wget -O plantvillage.zip https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip
!unzip -q plantvillage.zip -d ./dataset
print("PlantVillage dataset downloaded and extracted!")

In [ ]:
# 3. Clone / Setup Training Architecture & Class Taxonomy
import os
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-3

# Data Augmentation
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# 4. Locate dataset directory and split into Train / Validation sets
data_path = "./dataset/PlantVillage-Dataset-master/raw/color"
if not os.path.exists(data_path):
    # Fallback search
    for root, dirs, files in os.walk("./dataset"):
        if "color" in root.lower() or any("tomato" in d.lower() for d in dirs):
            data_path = root
            break

full_dataset = datasets.ImageFolder(data_path, transform=train_transform)
num_classes = len(full_dataset.classes)
print(f"Found {len(full_dataset)} images across {num_classes} classes!")

# 85% Train, 15% Validation split
train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# 5. Build Model Backbone (MobileNetV3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, num_classes)
model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
print(f"Model initialized and moved to {device}")

In [ ]:
# 6. Train and Validate Model
best_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * imgs.size(0)
        _, predicted = torch.max(preds, 1)
        train_correct += (predicted == targets).sum().item()
        train_total += targets.size(0)
        
    scheduler.step()
    
    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            preds = model(imgs)
            loss = criterion(preds, targets)
            val_loss += loss.item() * imgs.size(0)
            _, predicted = torch.max(preds, 1)
            val_correct += (predicted == targets).sum().item()
            val_total += targets.size(0)
            
    t_acc = train_correct / train_total * 100.0
    v_acc = val_correct / val_total * 100.0
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] Train Loss: {train_loss/train_total:.4f} (Acc: {t_acc:.2f}%) | Val Loss: {val_loss/val_total:.4f} (Acc: {v_acc:.2f}%)")
    
    if v_acc > best_acc:
        best_acc = v_acc
        torch.save(model.state_dict(), "model_weights.pth")
        print(f" >> Saved new best model checkpoint (Val Acc: {best_acc:.2f}%)")

print(f"\nTraining Complete! Best Accuracy: {best_acc:.2f}%")

In [ ]:
# 7. Download model_weights.pth to your computer
from google.colab import files
files.download("model_weights.pth")
print("Place the downloaded 'model_weights.pth' into your 'backend/' folder!")